<img src="https://www.plus2net.com/images/top2.jpg" alt="plus2net logo" style="height:40px;">

# Incremental (Delta) Append Pipeline for Parquet Data Lakes

This practice notebook accompanies the [Incremental Parquet Append Pipeline Tutorial on plus2net](https://www.plus2net.com/python/pandas-incremental-parquet-append.php).

**Objectives:**
1. Understand Parquet immutability and directory partitioning.
2. Append daily delta batches using `to_parquet()` with `partition_cols`.
3. Download sample student datasets using `!wget` and simulate incremental date appends.
4. Query data lakes with partition pruning filters (`filters=[...]`).
5. Execute self-validating assertions.

### Step 1: Install Dependencies

In [ ]:
# Install pyarrow and pandas
!pip install -q pyarrow pandas numpy

### Step 2: Download Benchmark Files

In [ ]:
# Download sample student dataset
!wget https://www.plus2net.com/python/download/student.csv

### Step 3: Initial Data Lake Partition Creation

In [ ]:
import pandas as pd

# Batch A (Nov 1, 2026)
df_batch_a = pd.read_csv('student.csv')
df_batch_a['entry_date'] = '2026-11-01'

# Write initial partition
df_batch_a.to_parquet('student_data_lake', partition_cols=['entry_date'], engine='pyarrow', index=False)
print("Initial batch written to student_data_lake!")

### Step 4: Incremental Delta Append (Nov 2, 2026)

In [ ]:
# Batch B (Nov 2, 2026)
df_batch_b = pd.DataFrame({
    'id': [101, 102],
    'name': ['Elena', 'Frank'],
    'class': ['Four', 'Five'],
    'mark': [88, 95],
    'sex': ['female', 'male'],
    'entry_date': ['2026-11-02', '2026-11-02']
})

# Append Batch B into same root directory without overwriting
df_batch_b.to_parquet('student_data_lake', partition_cols=['entry_date'], engine='pyarrow', index=False)
print("Incremental Delta Batch B appended successfully!")

### Step 5: Read Lake & Test Partition Pruning Filters

In [ ]:
# Read full lake
df_full = pd.read_parquet('student_data_lake')
print(f"Total Data Lake Records: {len(df_full)}")

# Selective read via partition pruning
df_nov2 = pd.read_parquet('student_data_lake', filters=[('entry_date', '==', '2026-11-02')])
print("\nNov 2 Partition Read:")
print(df_nov2)

### Step 6: Exercise Verification

In [ ]:
df_north = pd.DataFrame({"order_id": [1, 2], "amount": [150.0, 200.0], "region": ["North", "North"]})
df_north.to_parquet("sales_lake", partition_cols=["region"], index=False)

df_south = pd.DataFrame({"order_id": [3, 4, 5], "amount": [310.0, 420.0, 110.0], "region": ["South", "South", "South"]})
df_south.to_parquet("sales_lake", partition_cols=["region"], index=False)

df_south_read = pd.read_parquet("sales_lake", filters=[("region", "==", "South")])
assert len(df_south_read) == 3
print("Delta append exercise passed successfully!")

---
### Return to Tutorial
For more detailed tutorials and Pandas guides, visit [www.plus2net.com](https://www.plus2net.com/python/pandas-incremental-parquet-append.php).